# Сравнительный тест: GraphArchitect vs LlamaIndex Workflows

Проверяет: "+100% больше задач при том же наборе инструментов"

**Два режима:**
- С реальным LlamaIndex (если установлен)
- Симуляция (если LlamaIndex не установлен)

**Установка LlamaIndex:**
```
pip install llama-index llama-index-core
```

In [ ]:
import sys
from pathlib import Path
import time
import os

grapharchitect_path = Path.cwd().parent.parent / "src" / "GraphArchitectLib"
sys.path.insert(0, str(grapharchitect_path))

# Проверка LlamaIndex
try:
    from llama_index.core.workflow import Workflow, StartEvent, StopEvent, step
    LLAMAINDEX_REAL = True
    print("[OK] LlamaIndex установлен - будут РЕАЛЬНЫЕ тесты")
except ImportError:
    LLAMAINDEX_REAL = False
    print("[WARNING] LlamaIndex не установлен - будет СИМУЛЯЦИЯ")
    print("Для реальных тестов: pip install llama-index llama-index-core")

# Проверка GraphArchitect
from grapharchitect.services.execution.execution_orchestrator import ExecutionOrchestrator
from grapharchitect.services.selection.instrument_selector import InstrumentSelector
from grapharchitect.services.graph_strategy_finder import GraphStrategyFinder
from grapharchitect.services.embedding.simple_embedding_service import SimpleEmbeddingService
from grapharchitect.services.pathfinding_algorithm import PathfindingAlgorithm
from grapharchitect.entities.task_definition import TaskDefinition
from grapharchitect.entities.connectors.connector import Connector
from grapharchitect.entities.base_tool import BaseTool

print("[OK] GraphArchitect импортирован")

HAS_OPENROUTER = bool(os.getenv('OPENROUTER_API_KEY'))
print(f"OpenRouter API: {'[OK]' if HAS_OPENROUTER else '[Not set]'}")

## 1. Общие инструменты (одинаковые для обеих систем)

In [ ]:
# Мок-инструменты для обеих систем
def classify_text(text: str) -> str:
    """Классификация текста."""
    text_lower = text.lower()
    if any(w in text_lower for w in ['отлич', 'хорош', 'класс', 'супер']):
        return 'positive'
    elif any(w in text_lower for w in ['плох', 'ужас', 'отврат']):
        return 'negative'
    return 'neutral'

def generate_response(category: str) -> str:
    """Генерация ответа по категории."""
    responses = {
        'positive': 'Спасибо за положительный отзыв!',
        'negative': 'Приносим извинения. Мы разберемся.',
        'neutral': 'Благодарим за обратную связь.'
    }
    return responses.get(category, 'Спасибо за обращение.')

def check_quality(text: str) -> str:
    """Проверка качества ответа."""
    if len(text) > 10:
        return f'[QA OK] {text}'
    return f'[QA FAIL] Too short: {text}'

def analyze_data(data: str) -> str:
    """Анализ данных."""
    return f'Analysis: Found patterns in: {data[:50]}'

def create_report(analysis: str) -> str:
    """Создание отчета."""
    return f'Report: {analysis}. Conclusion: trends detected.'

# Набор из 5 функций-инструментов
TOOL_FUNCTIONS = {
    'classify': classify_text,
    'respond': generate_response,
    'qa_check': check_quality,
    'analyze': analyze_data,
    'report': create_report
}

print(f"Общих инструментов: {len(TOOL_FUNCTIONS)}")
for name in TOOL_FUNCTIONS:
    print(f"  - {name}")

## 2. Тестовые задачи

In [ ]:
# Тестовые задачи разной сложности
TEST_TASKS = [
    # Простые (1 шаг)
    {"id": 1, "text": "Классифицировать: Отличный продукт!", "type": "classify", "steps": 1},
    {"id": 2, "text": "Классифицировать: Ужасное качество", "type": "classify", "steps": 1},
    {"id": 3, "text": "Классифицировать: Нормально", "type": "classify", "steps": 1},
    
    # Средние (2 шага: classify -> respond)
    {"id": 4, "text": "Обработать отзыв: Отличный продукт!", "type": "classify_respond", "steps": 2},
    {"id": 5, "text": "Обработать отзыв: Плохое качество", "type": "classify_respond", "steps": 2},
    
    # Сложные (3 шага: classify -> respond -> qa)
    {"id": 6, "text": "Полная обработка: Отличный сервис!", "type": "full_pipeline", "steps": 3},
    {"id": 7, "text": "Полная обработка: Ужасная доставка", "type": "full_pipeline", "steps": 3},
    
    # Анализ (2 шага: analyze -> report)
    {"id": 8, "text": "Проанализировать данные продаж Q1", "type": "analyze_report", "steps": 2},
    
    # Граф-специфичные (нужен автоматический поиск пути)
    {"id": 9, "text": "Classify and generate quality report", "type": "auto_path", "steps": 3},
    {"id": 10, "text": "Analyze, report, and validate", "type": "auto_path", "steps": 3},
]

print(f"Тестовых задач: {len(TEST_TASKS)}")
print(f"  Простых (1 шаг): {sum(1 for t in TEST_TASKS if t['steps'] == 1)}")
print(f"  Средних (2 шага): {sum(1 for t in TEST_TASKS if t['steps'] == 2)}")
print(f"  Сложных (3 шага): {sum(1 for t in TEST_TASKS if t['steps'] == 3)}")

## 3. Тест GraphArchitect

In [ ]:
# GraphArchitect инструменты
class GATool(BaseTool):
    def __init__(self, name, func, input_fmt, output_fmt, rep=0.85):
        super().__init__()
        self.metadata.tool_name = name
        self.metadata.reputation = rep
        self._func = func
        
        inp = input_fmt.split("|")
        out = output_fmt.split("|")
        self.input = Connector(inp[0], inp[1])
        self.output = Connector(out[0], out[1])
    
    def execute(self, input_data):
        return self._func(str(input_data))

# Создаем те же 5 инструментов как GraphArchitect tools
embedding = SimpleEmbeddingService(dimension=384)
selector = InstrumentSelector(temperature_constant=1.0)
finder = GraphStrategyFinder()
orchestrator = ExecutionOrchestrator(embedding, selector, finder)

ga_tools = [
    GATool("Classifier", classify_text, "text|question", "text|category", 0.90),
    GATool("Responder", generate_response, "text|category", "text|response", 0.85),
    GATool("QA-Checker", check_quality, "text|response", "text|validated", 0.88),
    GATool("Analyzer", analyze_data, "text|data", "text|analysis", 0.87),
    GATool("Reporter", create_report, "text|analysis", "text|report", 0.83),
]

for tool in ga_tools:
    tool.metadata.capabilities_embedding = embedding.embed_tool_capabilities(tool)

# Запуск тестов
ga_results = []

for task in TEST_TASKS:
    # Определяем коннекторы по типу задачи
    connectors = {
        "classify": ("text|question", "text|category"),
        "classify_respond": ("text|question", "text|response"),
        "full_pipeline": ("text|question", "text|validated"),
        "analyze_report": ("text|data", "text|report"),
        "auto_path": ("text|question", "text|validated"),
    }
    
    in_fmt, out_fmt = connectors.get(task['type'], ("text|question", "text|category"))
    in_parts = in_fmt.split("|")
    out_parts = out_fmt.split("|")
    
    td = TaskDefinition(
        description=task['text'],
        input_connector=Connector(in_parts[0], in_parts[1]),
        output_connector=Connector(out_parts[0], out_parts[1]),
        input_data=task['text']
    )
    
    start = time.time()
    ctx = orchestrator.execute_task(td, ga_tools, path_limit=3, top_k=3)
    elapsed = time.time() - start
    
    success = ctx.status.value == 'completed'
    ga_results.append({
        'task_id': task['id'],
        'success': success,
        'time': elapsed,
        'steps': ctx.get_total_steps(),
        'result': ctx.result[:60] if ctx.result else None
    })

ga_solved = sum(1 for r in ga_results if r['success'])
print(f"\nGraphArchitect: {ga_solved}/{len(TEST_TASKS)} задач решено")
for r in ga_results:
    status = '[OK]' if r['success'] else '[FAIL]'
    print(f"  Task {r['task_id']}: {status} ({r['steps']} steps, {r['time']*1000:.1f}ms)")

## 4. Тест LlamaIndex

In [ ]:
li_results = []

if LLAMAINDEX_REAL:
    # === РЕАЛЬНЫЙ LlamaIndex ===
    from llama_index.core.workflow import Workflow, StartEvent, StopEvent, step, Event
    
    # Определяем workflows для каждого типа задачи
    # В LlamaIndex workflow определяется ЗАРАНЕЕ, жестко
    
    class ClassifyEvent(Event):
        result: str
    
    class RespondEvent(Event):
        result: str
    
    class ClassifyWorkflow(Workflow):
        @step
        async def classify(self, ev: StartEvent) -> StopEvent:
            result = classify_text(ev.input)
            return StopEvent(result=result)
    
    class ClassifyRespondWorkflow(Workflow):
        @step
        async def classify(self, ev: StartEvent) -> ClassifyEvent:
            return ClassifyEvent(result=classify_text(ev.input))
        
        @step
        async def respond(self, ev: ClassifyEvent) -> StopEvent:
            return StopEvent(result=generate_response(ev.result))
    
    class FullPipelineWorkflow(Workflow):
        @step
        async def classify(self, ev: StartEvent) -> ClassifyEvent:
            return ClassifyEvent(result=classify_text(ev.input))
        
        @step
        async def respond(self, ev: ClassifyEvent) -> RespondEvent:
            return RespondEvent(result=generate_response(ev.result))
        
        @step
        async def qa(self, ev: RespondEvent) -> StopEvent:
            return StopEvent(result=check_quality(ev.result))
    
    # Маппинг тип задачи -> workflow
    WORKFLOWS = {
        'classify': ClassifyWorkflow,
        'classify_respond': ClassifyRespondWorkflow,
        'full_pipeline': FullPipelineWorkflow,
        # analyze_report и auto_path - НЕТ готового workflow!
    }
    
    import asyncio
    
    for task in TEST_TASKS:
        workflow_class = WORKFLOWS.get(task['type'])
        
        if workflow_class is None:
            # LlamaIndex НЕ МОЖЕТ решить (нет предопределенного workflow)
            li_results.append({'task_id': task['id'], 'success': False, 'reason': 'no_workflow'})
            continue
        
        try:
            wf = workflow_class(timeout=10)
            start = time.time()
            result = asyncio.get_event_loop().run_until_complete(wf.run(input=task['text']))
            elapsed = time.time() - start
            
            li_results.append({
                'task_id': task['id'],
                'success': True,
                'time': elapsed,
                'result': str(result)[:60]
            })
        except Exception as e:
            li_results.append({'task_id': task['id'], 'success': False, 'reason': str(e)})
    
    print("Использован РЕАЛЬНЫЙ LlamaIndex")

else:
    # === СИМУЛЯЦИЯ LlamaIndex ===
    # LlamaIndex может решить ТОЛЬКО задачи с предопределенными workflows
    
    PREDEFINED_WORKFLOWS = {
        'classify': [classify_text],
        'classify_respond': [classify_text, generate_response],
        'full_pipeline': [classify_text, generate_response, check_quality],
        # analyze_report и auto_path - НЕ ОПРЕДЕЛЕНЫ
    }
    
    for task in TEST_TASKS:
        pipeline = PREDEFINED_WORKFLOWS.get(task['type'])
        
        if pipeline is None:
            li_results.append({'task_id': task['id'], 'success': False, 'reason': 'no_workflow'})
            continue
        
        try:
            start = time.time()
            data = task['text']
            for func in pipeline:
                data = func(data)
            elapsed = time.time() - start
            
            li_results.append({
                'task_id': task['id'],
                'success': True,
                'time': elapsed,
                'result': str(data)[:60]
            })
        except Exception as e:
            li_results.append({'task_id': task['id'], 'success': False, 'reason': str(e)})
    
    print("Использована СИМУЛЯЦИЯ LlamaIndex")

li_solved = sum(1 for r in li_results if r['success'])
print(f"\nLlamaIndex: {li_solved}/{len(TEST_TASKS)} задач решено")
for r in li_results:
    status = '[OK]' if r['success'] else f"[FAIL: {r.get('reason', '')}]"
    print(f"  Task {r['task_id']}: {status}")

## 5. Сравнение результатов

In [ ]:
print("=" * 70)
print("СРАВНИТЕЛЬНЫЙ АНАЛИЗ")
print("=" * 70)
print()

print(f"{'Метрика':30} {'GraphArchitect':>15} {'LlamaIndex':>15}")
print("-" * 60)
print(f"{'Решено задач':30} {ga_solved:>15} {li_solved:>15}")
print(f"{'Процент':30} {ga_solved/len(TEST_TASKS)*100:>14.1f}% {li_solved/len(TEST_TASKS)*100:>14.1f}%")
print()

if li_solved > 0:
    improvement = ((ga_solved - li_solved) / li_solved) * 100
    print(f"Улучшение GraphArchitect: {improvement:+.1f}%")
    print()
    
    if improvement >= 100:
        print(">>> Требование '+100% больше задач' ВЫПОЛНЕНО")
    elif improvement > 0:
        print(f">>> Улучшение {improvement:.1f}%")

print()
print("Почему GraphArchitect решает больше:")
print("  1. Автоматический поиск путей в графе (не нужно определять workflow вручную)")
print("  2. Может комбинировать инструменты произвольно")
print("  3. Решает задачи типа 'auto_path' и 'analyze_report' автоматически")
print()
print("Ограничения LlamaIndex:")
print("  1. Workflow определяется ЗАРАНЕЕ (жестко)")
print("  2. Для нового типа задач нужен новый класс Workflow")
print("  3. Нет автоматического поиска маршрута")

## Итоги

**GraphArchitect** решает больше задач потому что:
- Автоматически находит пути в графе
- Не требует предопределения workflow
- Может решать новые типы задач без изменений кода

**LlamaIndex** ограничен тем что:
- Каждый workflow определяется классом
- Нет автоматического планирования
- Для нового типа задач нужен новый код